# WideDepth datasets: Run-All introduction

This notebook is a small, executable tour of the [WideDepth benchmark](https://huggingface.co/datasets/IlyaInd/WideDepth) and [WideDepth train](https://huggingface.co/datasets/IlyaInd/WideDepth-train) releases. It downloads one verified sample from each Hugging Face dataset, demonstrates every public loader task, prints the array contracts, and visualizes the results.

Run every cell from top to bottom. The download cell is rerun-safe: Hugging Face reuses files that are already present.

## 1. Environment and configuration

Install the repository requirements into a clean environment before running the notebook. A dedicated kernel avoids binary conflicts between NumPy and compiled visualization packages:

```bash
python -m venv .venv
source .venv/bin/activate
python -m pip install -r requirements.txt ipykernel
python -m ipykernel install --user --name widedepth --display-name "Python (WideDepth)"
```

Select **Python (WideDepth)** as the notebook kernel. By default, only the ten PNG files needed for this tour are downloaded into the repository's ignored `downloads/quickstart` directory. To use an existing snapshot instead, set `WIDEDEPTH_BENCHMARK_ROOT` and/or `WIDEDEPTH_TRAIN_ROOT` before starting Jupyter. Each override must point directly to the directory containing the dataset's scene or modality folders.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from importlib.metadata import version
from pathlib import Path
import os
import sys

import numpy as np

numpy_major = int(np.__version__.split(".", maxsplit=1)[0])
matplotlib_parts = tuple(int(part) for part in version("matplotlib").split(".")[:2])
if numpy_major >= 2 and matplotlib_parts < (3, 9):
    raise RuntimeError(
        f"Incompatible notebook kernel: NumPy {np.__version__} with Matplotlib "
        f"{version('matplotlib')}. Create/select the Python (WideDepth) kernel "
        "described above, then restart Run All."
    )

# Xet initialization is expensive for only ten exact files. The ordinary
# HTTP path has lower startup overhead for this notebook.
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")

%matplotlib inline
import matplotlib.pyplot as plt
from huggingface_hub import hf_hub_download

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "widedepth").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "widedepth").is_dir():
    raise RuntimeError("Run this notebook from the WideDepth repository.")
sys.path.insert(0, str(PROJECT_ROOT))

AUTO_DOWNLOAD = True
DOWNLOAD_ROOT = PROJECT_ROOT / "downloads" / "quickstart"
BENCHMARK_ROOT_OVERRIDE = os.environ.get("WIDEDEPTH_BENCHMARK_ROOT")
TRAIN_ROOT_OVERRIDE = os.environ.get("WIDEDEPTH_TRAIN_ROOT")
RUN_RNG = np.random.default_rng()
SPARSE_SEED = int(RUN_RNG.integers(0, np.iinfo(np.uint32).max))

print(f"Repository: {PROJECT_ROOT}")
print(f"Automatic download: {AUTO_DOWNLOAD}")

## 2. Obtain the two tiny snapshots

Each rerun of this cell randomly selects from small HF-verified pools chosen for useful visualization. The resulting allow-lists remain exact, so only ten PNGs are downloaded rather than either full dataset. Rerun this cell and the cells below it whenever you want a different pair of samples.

In [ ]:
BENCHMARK_SCENE_POOL = (
    "005_059_090",
    "020_065_180",
    "035_073_180",
    "050_077_180",
    "065_082_000",
    "100_093_180",
)
TRAIN_SAMPLE_POOL = ("04784", "05695", "05756", "11369", "11775", "11896")

benchmark_scene_id = str(RUN_RNG.choice(BENCHMARK_SCENE_POOL))
train_sample_id = str(RUN_RNG.choice(TRAIN_SAMPLE_POOL))
benchmark_prefix = f"{benchmark_scene_id}/120FOV/65mm"
BENCHMARK_FILES = [
    f"{benchmark_prefix}/CENTER/pano_crop.png",
    f"{benchmark_prefix}/CENTER/depth/pano_crop.png",
    f"{benchmark_prefix}/CENTER/disparity/pano_crop.png",
    f"{benchmark_prefix}/DOWN/pano_crop.png",
    f"{benchmark_prefix}/RIGHT/pano_crop.png",
]
TRAIN_FILES = [
    f"equirect_up/{train_sample_id}.png",
    f"equirect_down_virt/{train_sample_id}.png",
    f"disp_equirect/{train_sample_id}.png",
    f"left_fisheye/{train_sample_id}.png",
    f"depth_fisheye/{train_sample_id}.png",
]

def prepare_root(override, repo_id, local_name, filenames):
    if override:
        root = Path(override).expanduser().resolve()
        if not root.is_dir():
            raise FileNotFoundError(f"Dataset root is not a directory: {root}")
        print(f"Using local override for {repo_id}: {root}")
        return root
    if not AUTO_DOWNLOAD:
        raise RuntimeError(
            f"Set a local root for {repo_id} or enable AUTO_DOWNLOAD."
        )
    root = DOWNLOAD_ROOT / local_name
    def download_file(filename):
        return hf_hub_download(
            repo_id=repo_id,
            repo_type="dataset",
            filename=filename,
            local_dir=root,
        )
    with ThreadPoolExecutor(max_workers=len(filenames)) as executor:
        list(executor.map(download_file, filenames))
    print(f"Prepared {repo_id}: {root}")
    return root

with ThreadPoolExecutor(max_workers=2) as executor:
    benchmark_future = executor.submit(
        prepare_root,
        BENCHMARK_ROOT_OVERRIDE,
        "IlyaInd/WideDepth",
        f"benchmark/{benchmark_scene_id}",
        BENCHMARK_FILES,
    )
    train_future = executor.submit(
        prepare_root,
        TRAIN_ROOT_OVERRIDE,
        "IlyaInd/WideDepth-train",
        f"train/{train_sample_id}",
        TRAIN_FILES,
    )
    benchmark_root = benchmark_future.result()
    train_root = train_future.result()
print(f"Random benchmark scene: {benchmark_scene_id}")
print(f"Random train sample: {train_sample_id}")

## 3. Load every public task

A dataset object represents one task and camera configuration. Samples are ordinary dictionaries of NumPy arrays—PyTorch is not required. Benchmark depth completion creates a sparse input from dense ground truth using a new seed for each notebook run; train sparse depth is measured LiDAR-derived data. The train stereo view is explicitly cropped and rotated to remove unused equirectangular regions and put its epipolar direction on horizontal display rows.

In [ ]:
from widedepth import WideDepthBenchmarkDataset, WideDepthTrainDataset

benchmark_datasets = {
    task: WideDepthBenchmarkDataset(
        benchmark_root,
        task=task,
        image_type="pano_crop",
        fov=120,
        baseline_mm=65,
        sparse_ratio=0.01,
        seed=SPARSE_SEED,
    )
    for task in (
        "mono",
        "depth_completion",
        "stereo_vertical",
        "stereo_horizontal",
    )
}
train_datasets = {
    "stereo": WideDepthTrainDataset(
        train_root,
        task="stereo",
        crop_equirect=True,
        rotate_ccw=True,
    ),
    "sparse_depth": WideDepthTrainDataset(train_root, task="sparse_depth"),
}

def choose_shared_id(datasets, id_attribute, preferred_id):
    shared_ids = set.intersection(
        *(set(getattr(dataset, id_attribute)) for dataset in datasets.values())
    )
    if not shared_ids:
        raise RuntimeError("The selected task snapshots have no shared sample ID.")
    return preferred_id if preferred_id in shared_ids else str(RUN_RNG.choice(sorted(shared_ids)))

def sample_with_id(dataset, id_attribute, sample_id):
    identifiers = getattr(dataset, id_attribute)
    return dataset[identifiers.index(sample_id)]

benchmark_scene_id = choose_shared_id(benchmark_datasets, "scene_ids", benchmark_scene_id)
train_sample_id = choose_shared_id(train_datasets, "sample_ids", train_sample_id)
benchmark_samples = {
    name: sample_with_id(dataset, "scene_ids", benchmark_scene_id)
    for name, dataset in benchmark_datasets.items()
}
train_samples = {
    name: sample_with_id(dataset, "sample_ids", train_sample_id)
    for name, dataset in train_datasets.items()
}
print(f"Loaded all six tasks: benchmark={benchmark_scene_id}, train={train_sample_id}.")

## 4. Inspect keys, shapes, dtypes, units, and validity

Images use RGB channel order and `uint8` HWC layout. Depth values are metres, disparities are pixels, and zero-valued target pixels are invalid.

In [ ]:
UNITS = {
    "depth": "metres",
    "sparse_depth": "metres",
    "disparity": "pixels",
    "valid_mask": "boolean",
}

def describe_sample(label, sample):
    print(f"\n{label}")
    print(f"  keys: {list(sample)}")
    for key, value in sample.items():
        if isinstance(value, np.ndarray):
            unit = UNITS.get(key, "RGB values" if value.ndim == 3 else "-")
            print(f"  {key:16s} shape={str(value.shape):18s} dtype={str(value.dtype):7s} unit={unit}")
    valid_mask = sample.get("valid_mask")
    if valid_mask is not None:
        print(f"  valid pixels: {valid_mask.mean():.2%}")

for task, sample in benchmark_samples.items():
    describe_sample(f"Benchmark / {task}", sample)
for task, sample in train_samples.items():
    describe_sample(f"Train / {task}", sample)

## 5. Visualize the benchmark sample

Scalar maps use robust 1st-to-99th-percentile limits so a few extreme pixels do not flatten useful contrast. Depth uses `Spectral`, disparity uses `turbo`, and black pixels are invalid zeros.

In [ ]:
def show_rgb(ax, image, title):
    ax.imshow(image)
    ax.set_title(title)
    ax.axis("off")

def show_scalar(ax, values, title, unit, cmap="Spectral"):
    valid = values > 0
    vmin = float(np.percentile(values[valid], 1)) if valid.any() else 0.0
    vmax = float(np.percentile(values[valid], 99)) if valid.any() else 1.0
    color_map = plt.get_cmap(cmap).copy()
    color_map.set_bad("black")
    rendered = np.ma.masked_where(~valid, values)
    artist = ax.imshow(rendered, cmap=color_map, vmin=vmin, vmax=vmax)
    ax.set_title(title)
    ax.axis("off")
    colorbar = ax.figure.colorbar(artist, ax=ax, fraction=0.046, pad=0.04)
    colorbar.set_label(unit)

def show_mask(ax, mask, title):
    ax.imshow(mask, cmap="gray", vmin=0, vmax=1)
    ax.set_title(title)
    ax.axis("off")

def show_rectified_rgb(ax, image, title):
    show_rgb(ax, image, title)
    for y_coordinate in np.linspace(0.12, 0.88, 5) * image.shape[0]:
        ax.axhline(y_coordinate, color="red", linewidth=1.2, alpha=0.9)

def show_sparse_overlay(ax, image, values, title, unit, cmap, max_points=30_000):
    ax.imshow(image)
    valid_mask = values > 0
    y_coordinates, x_coordinates = np.nonzero(valid_mask)
    if not len(x_coordinates):
        raise ValueError(f"Cannot visualize {title}: the sample has no valid points.")
    stride = max(1, int(np.ceil(len(x_coordinates) / max_points)))
    y_coordinates = y_coordinates[::stride]
    x_coordinates = x_coordinates[::stride]
    point_values = values[y_coordinates, x_coordinates]
    vmin, vmax = np.percentile(point_values, [1, 99])
    artist = ax.scatter(
        x_coordinates,
        y_coordinates,
        c=point_values,
        s=4,
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        linewidths=0,
    )
    ax.set_title(title)
    ax.axis("off")
    colorbar = ax.figure.colorbar(artist, ax=ax, fraction=0.046, pad=0.04)
    colorbar.set_label(unit)

figure, axes = plt.subplots(3, 3, figsize=(15, 15), constrained_layout=True)

sample = benchmark_samples["mono"]
show_rgb(axes[0, 0], sample["image"], "Mono: RGB")
show_scalar(axes[0, 1], sample["depth"], "Mono: dense depth", "metres")
show_mask(axes[0, 2], sample["valid_mask"], "Mono: valid mask")

for row, task, label in (
    (1, "stereo_vertical", "Vertical stereo"),
    (2, "stereo_horizontal", "Horizontal stereo"),
):
    sample = benchmark_samples[task]
    show_rgb(axes[row, 0], sample["left_image"], f"{label}: left")
    show_rgb(axes[row, 1], sample["right_image"], f"{label}: right")
    show_scalar(axes[row, 2], sample["disparity"], f"{label}: disparity", "pixels", cmap="turbo")

figure.suptitle(f"WideDepth benchmark — scene {benchmark_scene_id}", fontsize=16)
plt.show()

## 6. Visualize the train sample

The stereo task uses aligned equirectangular views and encoded disparity. For display, both views are rotated counter-clockwise and overlaid with identical red horizontal guides, making their rectification along equirectangular meridians easier to inspect. The sparse-depth task uses the left fisheye RGB image and independently projected LiDAR depth; colored LiDAR points are overlaid on RGB so the measurements remain visible.

In [ ]:
stereo_figure, stereo_axes = plt.subplots(1, 3, figsize=(15, 10), constrained_layout=True)
sample = train_samples["stereo"]
show_rectified_rgb(stereo_axes[0], sample["left_image"], "Upper view — cropped and rotated")
show_rectified_rgb(stereo_axes[1], sample["right_image"], "Lower virtual view — cropped and rotated")
show_sparse_overlay(
    stereo_axes[2],
    sample["left_image"],
    sample["disparity"],
    "Disparity on upper view",
    "pixels",
    "turbo",
)
stereo_figure.suptitle(f"Train stereo — sample {train_sample_id}", fontsize=16)
plt.show()

sparse_figure, sparse_axes = plt.subplots(1, 2, figsize=(15, 6), constrained_layout=True)
sample = train_samples["sparse_depth"]
show_rgb(sparse_axes[0], sample["image"], "Fisheye RGB")
show_sparse_overlay(
    sparse_axes[1],
    sample["image"],
    sample["sparse_depth"],
    "Projected LiDAR depth points",
    "metres",
    "Spectral",
)
sparse_figure.suptitle(f"WideDepth train sparse depth — sample {train_sample_id}", fontsize=16)
plt.show()